In [ ]:
import random

class Environment:
    def __init__(self, state='Dirty'):
        self.state = state

    def get_percept(self):
        return self.state

    def clean_room(self):
        self.state = 'Clean'
        return 10

    def no_action_reward(self):
        return 0

class LearningBasedAgent:
    def __init__(self, actions):
        self.Q = {}
        self.actions = actions
        self.alpha = 0.1     # Learning rate
        self.gamma = 0.9     # Discount factor
        self.epsilon = 0.1   # Exploration rate (10% chance to act randomly)

    def get_Q_value(self, state, action):
        # Returns the Q-value if it exists, otherwise defaults to 0.0
        return self.Q.get((state, action), 0.0)

    def select_action(self, state):
        # Epsilon-Greedy approach: Explore vs Exploit   
        if random.uniform(0, 1) < self.epsilon:
            return random.choice(self.actions) # Explore: Pick a random action
        else:
            # Exploit: Pick the action with the highest Q-value for this state
            return max(self.actions, key=lambda a: self.get_Q_value(state, a))
 
    def learn(self, state, action, reward, next_state):
        # The Q-Learning formula
        old_Q = self.get_Q_value(state, action)
        best_future_Q = max([self.get_Q_value(next_state, a) for a in self.actions])
        
        # Calculate the new score and save it in the Q-tabl
        new_Q = old_Q + self.alpha * (reward + self.gamma * best_future_Q - old_Q)
        self.Q[(state, action)] = new_Q

    def act(self, state):
        return self.select_action(state)

def run_agent(agent, environment, steps):
    for step in range(steps):
        percept = environment.get_percept()
        action = agent.act(percept)
        
        # --- THE FIX ---
        # The environment only cleans and rewards IF the agent actually chose to clean
        if action == 'Clean the room' and percept == 'Dirty':
            reward = environment.clean_room()
        else:
            # If the agent does nothing, or tries to clean an already clean room
            reward = environment.no_action_reward()
        # ---------------
            
        print(f"Step {step + 1}: Percept - '{percept}', Action - '{action}', Reward - {reward}")
        
        next_percept = environment.get_percept()
        
        # The agent updates its brain based on what just happened
        agent.learn(percept, action, reward, next_percept)


# --- Execution ---

# Define the possible actions
actions = ['Clean the room', 'No action needed']

# Create instances of agent and environment
agent = LearningBasedAgent(actions)
environment = Environment()

# Run the agent in the environment for 10 steps 
# (Increased from 5 to give it a tiny bit more time to learn)
print("--- Starting Simulation ---")
run_agent(agent, environment, 10)

# Print the agent's memory (Q-table) to see what it learned
print("\n--- Agent's Q-Table (Memory) ---")
if not agent.Q:
    print("Q-table is empty.")
else:
    for state_action_pair, score in agent.Q.items():
        state = state_action_pair[0]
        action = state_action_pair[1]
        print(f"When '{state}', doing '{action}' has a score of: {score:.4f}")

--- Starting Simulation ---
Steps: 1 -> Action: Clean the room -> Percept: Dirty -> Reward: 10
Steps: 2 -> Action: No action needed -> Percept: Clean -> Reward: 0
Steps: 3 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 4 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 5 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 6 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 7 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 8 -> Action: No action needed -> Percept: Clean -> Reward: 0
Steps: 9 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 10 -> Action: Clean the room -> Percept: Clean -> Reward: 0

--- Agent's Q-Table (Memory) ---
When Dirty doing Clean the room has a score of 1.0000
When Clean doing No action needed has a score of 0.0000
When Clean doing Clean the room has a score of 0.0000


GEMINI's Explanation

Which made it easier for me to understand

Gemini said
Here is a step-by-step breakdown of the code. Let's demystify how this AI learns.

High-Level Overview
What is this program trying to do?
This program simulates a tiny, unprogrammed robot (the Agent) placed inside a room (the Environment). The robot's goal is to keep the room clean, but it hasn't been given any instructions on how to do that. It has to learn entirely through trial and error.

What problem is being solved?
In traditional programming, you write strict rules: if room == 'Dirty': clean(). In Artificial Intelligence, specifically Reinforcement Learning, we solve the problem of how a machine can teach itself without explicit rules. It solves this by exploring, making mistakes, and earning "points" (rewards) when it accidentally does the right thing.

Gemini said
Here is a step-by-step breakdown of the code. Let's demystify how this AI learns.

High-Level Overview
What is this program trying to do?
This program simulates a tiny, unprogrammed robot (the Agent) placed inside a room (the Environment). The robot's goal is to keep the room clean, but it hasn't been given any instructions on how to do that. It has to learn entirely through trial and error.

What problem is being solved?
In traditional programming, you write strict rules: if room == 'Dirty': clean(). In Artificial Intelligence, specifically Reinforcement Learning, we solve the problem of how a machine can teach itself without explicit rules. It solves this by exploring, making mistakes, and earning "points" (rewards) when it accidentally does the right thing.

Environment Class
This class represents the "world" the robot lives in. It's a very simple one-room world.

def __init__(self, state='Dirty'):

When the world is created, it starts in a 'Dirty' state by default.

def get_percept(self):

Returns the current condition of the room.

def clean_room(self):

If this is called, the room changes to 'Clean', and the environment spits out a reward of 10 points.

def no_action_reward(self):

If nothing useful happens, the environment gives a reward of 0 points.

What is "state" and "percept" here?

State: The actual, physical reality of the room (it is either Dirty or Clean).

Percept: What the robot's sensors "perceive" or see. In this simple code, the robot sees reality perfectly, so the percept and the state are the exact same thing.

Analogy: The "state" is that it is raining outside. The "percept" is you looking through a window and seeing the rain.


LearningBasedAgent Class (VERY DETAILED)
What is Q-learning in simple terms?
Q-learning is a method where an AI learns by keeping a "cheat sheet" (called a Q-table). Every time it tries an action in a specific situation, it scores how good or bad the result was and writes it down. Next time it faces that situation, it checks the cheat sheet to pick the action with the highest score.

What is the Q-table (self.Q) and how it works?
self.Q is a Python dictionary acting as the robot's brain/cheat sheet. It starts completely empty ({}). As the robot explores, it adds entries. An entry looks like this: ('Dirty', 'Clean the room'): 1.0. This translates to: "When I see a Dirty room, cleaning it has a score of 1.0."

Explaining the Variables:

self.alpha = 0.1 (Learning Rate): How quickly the robot overrides old beliefs with new information. At 0.1, it only changes its mind by 10% each time. Analogy: If you have a favorite restaurant but have one bad meal there, you don't immediately hate the place. You just lower your opinion of it slightly (10%).

self.gamma = 0.9 (Discount Factor): How much the robot cares about the future versus the present. At 0.9, it cares highly about long-term success, not just immediate points. Analogy: Choosing to study for an exam (future reward) instead of playing video games (immediate reward).

self.epsilon = 0.1 (Exploration Rate): The percentage of the time the robot will do something totally random just to see what happens. Here, there is a 10% chance it acts randomly. Analogy: 90% of the time you order your favorite pizza (Exploit), but 10% of the time you force yourself to try a weird new flavor just in case it's better (Explore).

Gemini said
Here is a step-by-step breakdown of the code. Let's demystify how this AI learns.

High-Level Overview
What is this program trying to do?
This program simulates a tiny, unprogrammed robot (the Agent) placed inside a room (the Environment). The robot's goal is to keep the room clean, but it hasn't been given any instructions on how to do that. It has to learn entirely through trial and error.

What problem is being solved?
In traditional programming, you write strict rules: if room == 'Dirty': clean(). In Artificial Intelligence, specifically Reinforcement Learning, we solve the problem of how a machine can teach itself without explicit rules. It solves this by exploring, making mistakes, and earning "points" (rewards) when it accidentally does the right thing.

Environment Class
This class represents the "world" the robot lives in. It's a very simple one-room world.

def __init__(self, state='Dirty'):

When the world is created, it starts in a 'Dirty' state by default.

def get_percept(self):

Returns the current condition of the room.

def clean_room(self):

If this is called, the room changes to 'Clean', and the environment spits out a reward of 10 points.

def no_action_reward(self):

If nothing useful happens, the environment gives a reward of 0 points.

What is "state" and "percept" here?

State: The actual, physical reality of the room (it is either Dirty or Clean).

Percept: What the robot's sensors "perceive" or see. In this simple code, the robot sees reality perfectly, so the percept and the state are the exact same thing.

Analogy: The "state" is that it is raining outside. The "percept" is you looking through a window and seeing the rain.

LearningBasedAgent Class (VERY DETAILED)
What is Q-learning in simple terms?
Q-learning is a method where an AI learns by keeping a "cheat sheet" (called a Q-table). Every time it tries an action in a specific situation, it scores how good or bad the result was and writes it down. Next time it faces that situation, it checks the cheat sheet to pick the action with the highest score.

What is the Q-table (self.Q) and how it works?
self.Q is a Python dictionary acting as the robot's brain/cheat sheet. It starts completely empty ({}). As the robot explores, it adds entries. An entry looks like this: ('Dirty', 'Clean the room'): 1.0. This translates to: "When I see a Dirty room, cleaning it has a score of 1.0."

Explaining the Variables:

self.alpha = 0.1 (Learning Rate): How quickly the robot overrides old beliefs with new information. At 0.1, it only changes its mind by 10% each time. Analogy: If you have a favorite restaurant but have one bad meal there, you don't immediately hate the place. You just lower your opinion of it slightly (10%).

self.gamma = 0.9 (Discount Factor): How much the robot cares about the future versus the present. At 0.9, it cares highly about long-term success, not just immediate points. Analogy: Choosing to study for an exam (future reward) instead of playing video games (immediate reward).

self.epsilon = 0.1 (Exploration Rate): The percentage of the time the robot will do something totally random just to see what happens. Here, there is a 10% chance it acts randomly. Analogy: 90% of the time you order your favorite pizza (Exploit), but 10% of the time you force yourself to try a weird new flavor just in case it's better (Explore).

Explaining the Methods:

def get_Q_value(self, state, action):

The robot opens its cheat sheet and looks up the score for a specific action. The get(..., 0.0) part means if it has never tried this action before, it assumes the score is 0.0.

def select_action(self, state):

This is the Epsilon-Greedy approach in action.

if random.uniform(0, 1) < self.epsilon: The robot rolls a 100-sided die. If it lands on 1-10, it picks an action completely at random.

else: If it lands on 11-100, it looks at its cheat sheet (self.Q) and picks the action with the absolute highest score for the current state.

def learn(self, state, action, reward, next_state):

This is where the magic happens. The robot updates its cheat sheet.

old_Q: "What did I think the score was before I took this action?"

best_future_Q: "Now that I took the action and the room changed, what is the best possible score I can get on my next turn?"

The Formula: new_Q=old_Q+α×(reward+γ×best_future_Q−old_Q)

Simple English translation: "Take my old score, and nudge it a little bit (alpha) toward the reality of the reward I just got, plus the future points I expect to get."

def act(self, state):

A simple helper function that just triggers the select_action process.

Small Example of the Q-Table Changing:

Robot sees 'Dirty'. Chooses 'Clean'. Cheat sheet says old score is 0.0.

Gets +10 reward. Room is now 'Clean'.

Formula calculates: 0.0 + 0.1 * (10 + 0.9 * 0.0 - 0.0).

New score is 1.0. The cheat sheet is updated: ('Dirty', 'Clean the room') = 1.0.


DETAILED EXPLANATION OF THE AGENT CLASS'S METHODS 

1) def get_Q_value(self, state, action): 

The Big Picture
Think of self.Q as the robot’s Cheat Sheet (technically, it's a Python dictionary).

Imagine a literal notebook. On the left side of the page, the robot writes down a specific situation. On the right side, it writes down the score for that situation.

1. What is (state, action)?
This is the specific situation the robot is trying to look up in its notebook. In Python, putting things in parentheses like this creates a Tuple (just a grouped package of items).

For example, if the room is Dirty and the robot is thinking about cleaning it, the tuple looks like this: ('Dirty', 'Clean the room').

This group of two words acts as the Search Key the robot uses to flip through its notebook.

2. What is self.Q.get(...)?
Because self.Q is a Python dictionary, it comes with a built-in superpower called .get().

Normally in Python, if you ask a dictionary to look up a word that isn't in there yet, the whole program crashes and throws a massive error.

.get() is the polite, safe way to check a dictionary. It translates to: "Hey, check if you have this label. If you do, give me the value. If you don't, please don't crash, just give me a backup answer instead."

3. What is the 0.0?
That 0.0 is the backup answer (the default value).

When the robot first turns on, its notebook (self.Q) is completely empty. It has never experienced anything.

So, if the robot asks its notebook: "Hey, what is the score for ('Dirty', 'Clean the room')?" The notebook says: "I have never seen that in my life. I don't have a score for it. So, I will just give you the default backup score of 0.0."


"Hey Notebook (self.Q), please safely look up (.get) the current score for this exact situation ((state, action)). If you have a score written down, hand it to me. But if this page is completely blank because we haven't learned this yet, don't panic—just hand me a 0.0 so I know we are starting from scratch."

Why is this necessary?
Because the Q-learning math formula from the other function needs a number to do its math. If the agent is trying something for the very first time, it needs a starting point. 0.0 tells the math formula, "We have zero expectations about whether this is a good or bad idea yet. Let's try it and find out!"

2) def select_action(self, state):

The Big Picture: The "Explore vs. Exploit" Dilemma
This function answers one simple question for the robot: "How do I decide what to do right now?"

Imagine you are going out for dinner. You have two choices:

Exploit (Play it safe): Go to your absolute favorite restaurant where you know you will get a great meal.

Explore (Take a risk): Try that brand new weird restaurant down the street. It might be terrible, but it might become your new favorite!

If you always Exploit, you never discover anything new. If you always Explore, you eat a lot of terrible food.

This code gives the robot a mathematical way to balance trying new things (Exploring) with doing what it already knows works best (Exploiting).

Line-by-Line Breakdown
1. The Coin Toss (Rolling the Digital Die)
Python
if random.uniform(0, 1) < self.epsilon:
random.uniform(0, 1): This tells Python to pick a totally random decimal number between 0 and 1 (like 0.05, 0.43, 0.89, etc.). Imagine the robot is rolling a 100-sided die.

< self.epsilon: Remember from the setup that self.epsilon is 0.1.

Translation: "If my random number is less than 0.1 (which will happen exactly 10% of the time), do the next line."

2. The "Explore" Action
Python
    return random.choice(self.actions)
If the robot rolled a number under 0.1, it enters "Explore Mode".

random.choice literally puts all possible actions (Clean, Do Nothing) into a hat, closes its eyes, and pulls one out at random. It ignores the notebook completely.

3. The "Exploit" Action (The Tricky Part)
Python
else:
    return max(self.actions, key=lambda a: self.get_Q_value(state, a))
If the robot rolled a number 0.1 or higher (which happens 90% of the time), it enters "Exploit Mode". It wants to make the smartest choice possible based on what it has learned.

max(...): This means "Find me the biggest/best one."

self.actions: This is the list of things the robot can do: ['Clean the room', 'No action needed'].

Now, what is key=lambda a: self.get_Q_value(...)?
If you tell Python to find the max() of the words 'Clean' and 'Do nothing', Python gets confused. It asks, "How do you want me to judge them? By alphabetical order? By word length?"

key= is you telling Python the "rule" for judging.

lambda a: is just programmer shorthand for a quick, temporary rule. Treat a as a stand-in for "each action".

Translation of the whole line: "Look at my list of actions. To figure out which one is the max (the best), take each action (a), flip open my notebook (get_Q_value), and check its score for the current state. Whichever action has the highest score in the notebook, pick that one."

3) def learn(self, state, action, reward, next_state):

To make sense of this, let's step away from the robot vacuum for a second. Let's think about this formula exactly like evaluating a financial investment.

Imagine you are tracking a specific company's stock. You have an expectation of what it's worth, but reality keeps updating, so you have to constantly adjust your valuation.

Here is exactly what those four lines of code are doing.

Line 1: Checking Your Current Belief
Python
old_Q = self.get_Q_value(state, action)
What it means: "Before I see what happened today, what did my notebook say this investment was worth?"

The Analogy: You look at your spreadsheet. Before the market opened, you valued buying this company's stock (action) under current market conditions (state) at a score of 50.0 (old_Q).

Line 2: Looking Ahead at the New Reality
Python
best_future_Q = max([self.get_Q_value(next_state, a) for a in self.actions])
What it means: "Now that I took the action, the world has changed to a next_state. From this new position, what is the absolute best move I can make next, and how much is that worth?"

The Analogy: You bought the stock, and the company just announced a new product. That is your next_state. You look at your options going forward (hold, buy more, sell). You realize the best possible future move has a projected value of 60.0. That is your best_future_Q.

Gemini said
This is the absolute hardest part of the entire code, so do not feel bad for being stuck here! You are looking at a simplified version of the Bellman Equation. It is the mathematical engine that drives almost all modern Reinforcement Learning.

To make sense of this, let's step away from the robot vacuum for a second. Let's think about this formula exactly like evaluating a financial investment.

Imagine you are tracking a specific company's stock. You have an expectation of what it's worth, but reality keeps updating, so you have to constantly adjust your valuation.

Here is exactly what those four lines of code are doing.

Line 1: Checking Your Current Belief
Python
old_Q = self.get_Q_value(state, action)
What it means: "Before I see what happened today, what did my notebook say this investment was worth?"

The Analogy: You look at your spreadsheet. Before the market opened, you valued buying this company's stock (action) under current market conditions (state) at a score of 50.0 (old_Q).

Line 2: Looking Ahead at the New Reality
Python
best_future_Q = max([self.get_Q_value(next_state, a) for a in self.actions])
What it means: "Now that I took the action, the world has changed to a next_state. From this new position, what is the absolute best move I can make next, and how much is that worth?"

The Analogy: You bought the stock, and the company just announced a new product. That is your next_state. You look at your options going forward (hold, buy more, sell). You realize the best possible future move has a projected value of 60.0. That is your best_future_Q.

Line 3: The Reality Check (The Formula)
Python
new_Q = old_Q + self.alpha * (reward + self.gamma * best_future_Q - old_Q)
This looks terrifying, but it is just a formula for calculating "How much should I change my mind based on what just happened?"

Let's break the math down from the inside out:

Part A: The "Target Reality" (reward + self.gamma * best_future_Q)
This is what actually happened.

reward: The immediate cash dividend the stock paid you today (let's say 10 points).

self.gamma * best_future_Q: Your projected future value (60.0), multiplied by gamma (0.9). Gamma represents the idea that a dollar tomorrow is worth slightly less than a dollar today. So, 60.0×0.9=54.0.

Target Reality: 10+54.0=64.0. Based on reality, the stock should be worth 64.0.

Part B: The "Surprise Factor" (Target Reality - old_Q)

You thought it was worth 50.0 (old_Q).

Reality says it's worth 64.0.

64.0−50.0=+14.0.

You were pleasantly surprised by 14 points! If reality was worse than you expected, this number would be negative (a disappointment).

Part C: The "Adjustment" (self.alpha * Surprise Factor)
You don't just blindly change your valuation to 64.0 based on one good day. You are cautious. alpha (0.1) is your caution level.

You only adjust your belief by 10% of the surprise.

0.1×14.0=+1.4.

The Final Math:

new_Q = old_Q + Adjustment

new_Q = 50.0 + 1.4 = 51.4.

Line 4: Updating the Ledger
Python
self.Q[(state, action)] = new_Q
What it means: You take your eraser, rub out the old score of 50.0 in your notebook, and write down the new score of 51.4.

Next time you are in this exact situation, you will start with a slightly higher, more accurate expectation!